## การฝังตัว

ในตัวอย่างก่อนหน้านี้ เราดำเนินการบนเวกเตอร์ถุงคำที่มีมิติสูงโดยมีความยาว `vocab_size` และเราได้แปลงอย่างชัดเจนจากเวกเตอร์แทนตำแหน่งที่มีมิติต่ำเป็นเวกเตอร์หนึ่งร้อนแบบกระจาย การแทนแบบหนึ่งร้อนนี้ไม่ประหยัดหน่วยความจำ นอกจากนี้ แต่ละคำยังถูกปฏิบัติแยกจากกัน กล่าวคือ เวกเตอร์ที่เข้ารหัสแบบหนึ่งร้อนไม่แสดงความคล้ายคลึงทางความหมายระหว่างคำ

ในหน่วยนี้ เราจะเรียนรู้เพิ่มเติมเกี่ยวกับชุดข้อมูล **News AG** เพื่อเริ่มต้น มาโหลดข้อมูลและดึงคำนิยามบางอย่างจากโน้ตบุ๊กก่อนหน้า


In [1]:
import torch
import torchtext
import numpy as np
from torchnlp import *
train_dataset, test_dataset, classes, vocab = load_dataset()
vocab_size = len(vocab)
print("Vocab size = ",vocab_size)

Loading dataset...


d:\WORK\ai-for-beginners\5-NLP\14-Embeddings\data\train.csv: 29.5MB [00:01, 18.8MB/s]                            
d:\WORK\ai-for-beginners\5-NLP\14-Embeddings\data\test.csv: 1.86MB [00:00, 11.2MB/s]                          


Building vocab...
Vocab size =  95812



## What is embedding?

The idea of **embedding** is to represent words by lower-dimensional dense vectors, which somehow reflect semantic meaning of a word. We will later discuss how to build meaningful word embeddings, but for now let's just think of embeddings as a way to lower dimensionality of a word vector. 

So, embedding layer would take a word as an input, and produce an output vector of specified `embedding_size`. In a sense, it is very similar to `Linear` layer, but instead of taking one-hot encoded vector, it will be able to take a word number as an input.

By using embedding layer as a first layer in our network, we can switch from bag-of-words to **embedding bag** model, where we first convert each word in our text into corresponding embedding, and then compute some aggregate function over all those embeddings, such as `sum`, `average` or `max`.  

![Image showing an embedding classifier for five sequence words.](../../../../../translated_images/th/embedding-classifier-example.b77f021a7ee67eee.webp)

Our classifier neural network will start with embedding layer, then aggregation layer, and linear classifier on top of it:


In [2]:
class EmbedClassifier(torch.nn.Module):
    def __init__(self, vocab_size, embed_dim, num_class):
        super().__init__()
        self.embedding = torch.nn.Embedding(vocab_size, embed_dim)
        self.fc = torch.nn.Linear(embed_dim, num_class)

    def forward(self, x):
        x = self.embedding(x)
        x = torch.mean(x,dim=1)
        return self.fc(x)

### การจัดการกับขนาดลำดับตัวแปร

เป็นผลมาจากสถาปัตยกรรมนี้ มินิบาตช์ที่ส่งเข้าเครือข่ายของเราจำเป็นต้องถูกสร้างขึ้นในลักษณะเฉพาะ ในยูนิตก่อนหน้าเมื่อใช้ bag-of-words เทนเซอร์ BoW ทั้งหมดในมินิบาตช์จะมีขนาดเท่ากัน `vocab_size` ไม่ว่าจะมีความยาวลำดับข้อความจริงเท่าใดก็ตาม เมื่อเราเปลี่ยนมาใช้ word embeddings เราจะมีจำนวนคำที่แปรผันในแต่ละตัวอย่างข้อความ และเมื่อรวมตัวอย่างเหล่านั้นเป็นมินิบาตช์ เราจะต้องใช้การเติมเต็มบางอย่าง

สิ่งนี้สามารถทำได้โดยใช้เทคนิคเดียวกันในการให้ฟังก์ชัน `collate_fn` กับ datasource:


In [3]:
def padify(b):
    # b is the list of tuples of length batch_size
    #   - first element of a tuple = label, 
    #   - second = feature (text sequence)
    # build vectorized sequence
    v = [encode(x[1]) for x in b]
    # first, compute max length of a sequence in this minibatch
    l = max(map(len,v))
    return ( # tuple of two tensors - labels and features
        torch.LongTensor([t[0]-1 for t in b]),
        torch.stack([torch.nn.functional.pad(torch.tensor(t),(0,l-len(t)),mode='constant',value=0) for t in v])
    )

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=16, collate_fn=padify, shuffle=True)

### การฝึกอบรมตัวจำแนก embedding

ตอนนี้ที่เราได้กำหนด dataloader ที่เหมาะสมแล้ว เราสามารถฝึกอบรมโมเดลโดยใช้ฟังก์ชันการฝึกอบรมที่เราได้กำหนดในหน่วยก่อนหน้า:


In [4]:
net = EmbedClassifier(vocab_size,32,len(classes)).to(device)
train_epoch(net,train_loader, lr=1, epoch_size=25000)

3200: acc=0.6415625
6400: acc=0.6865625
9600: acc=0.7103125
12800: acc=0.726953125
16000: acc=0.739375
19200: acc=0.75046875
22400: acc=0.7572321428571429


(0.889799795315499, 0.7623160588611644)

> **หมายเหตุ**: เรากำลังฝึกเพียง 25k เรคอร์ดเท่านั้นที่นี่ (น้อยกว่าหนึ่งยุคเต็ม) เพื่อประหยัดเวลา แต่คุณสามารถฝึกต่อไป เขียนฟังก์ชันเพื่อฝึกหลายยุค และทดลองกับพารามิเตอร์อัตราการเรียนรู้เพื่อให้ได้ความแม่นยำสูงขึ้น คุณควรสามารถไปถึงความแม่นยำประมาณ 90% ได้


### ชั้น EmbeddingBag และการแทนลำดับความยาวตัวแปร

ในสถาปัตยกรรมก่อนหน้า เราจำเป็นต้องเติมเต็ม (pad) ลำดับทั้งหมดให้มีความยาวเท่ากันเพื่อที่จะสามารถจัดใส่ในมินิบัตช์ได้ วิธีนี้ไม่ใช่วิธีที่มีประสิทธิภาพที่สุดสำหรับการแทนลำดับความยาวตัวแปร - อีกวิธีหนึ่งคือการใช้เวกเตอร์ **offset** ซึ่งจะเก็บออฟเซ็ตของลำดับทั้งหมดที่จัดเก็บในเวกเตอร์ขนาดใหญ่ชุดเดียว

![ภาพแสดงการแทนลำดับด้วยออฟเซ็ต](../../../../../translated_images/th/offset-sequence-representation.eb73fcefb29b46ee.webp)

> **หมายเหตุ**: ในภาพด้านบนนี้ เราแสดงลำดับของอักขระ แต่ในตัวอย่างของเราเรากำลังทำงานกับลำดับของคำ อย่างไรก็ตาม หลักการทั่วไปของการแทนลำดับด้วยเวกเตอร์ออฟเซ็ตยังคงเหมือนเดิม

เพื่อทำงานกับการแสดงออฟเซ็ต เราใช้เลเยอร์ [`EmbeddingBag`](https://pytorch.org/docs/stable/generated/torch.nn.EmbeddingBag.html) ซึ่งคล้ายกับ `Embedding` แต่จะรับเวกเตอร์เนื้อหาและเวกเตอร์ออฟเซ็ตเป็นอินพุต และยังรวมถึงเลเยอร์เฉลี่ยซึ่งสามารถเป็น `mean`, `sum` หรือ `max` ได้

นี่คือตัวอย่างเครือข่ายที่แก้ไขแล้วซึ่งใช้ `EmbeddingBag`:


In [5]:
class EmbedClassifier(torch.nn.Module):
    def __init__(self, vocab_size, embed_dim, num_class):
        super().__init__()
        self.embedding = torch.nn.EmbeddingBag(vocab_size, embed_dim)
        self.fc = torch.nn.Linear(embed_dim, num_class)

    def forward(self, text, off):
        x = self.embedding(text, off)
        return self.fc(x)

ในการเตรียมชุดข้อมูลสำหรับการฝึก เราจำเป็นต้องเตรียมฟังก์ชันการแปลงที่จะเตรียมเวกเตอร์ออฟเซ็ต:


In [6]:
def offsetify(b):
    # first, compute data tensor from all sequences
    x = [torch.tensor(encode(t[1])) for t in b]
    # now, compute the offsets by accumulating the tensor of sequence lengths
    o = [0] + [len(t) for t in x]
    o = torch.tensor(o[:-1]).cumsum(dim=0)
    return ( 
        torch.LongTensor([t[0]-1 for t in b]), # labels
        torch.cat(x), # text 
        o
    )

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=16, collate_fn=offsetify, shuffle=True)

โปรดทราบว่า แตกต่างจากตัวอย่างก่อนหน้านี้ทั้งหมด เครือข่ายของเราขณะนี้รับพารามิเตอร์สองตัว: เวกเตอร์ข้อมูลและเวกเตอร์ออฟเซ็ต ซึ่งมีขนาดต่างกัน เช่นเดียวกัน ตัวโหลดข้อมูลของเรายังให้ค่า 3 ค่าแทนที่จะเป็น 2: เวกเตอร์ทั้งข้อความและออฟเซ็ตถูกจัดเตรียมเป็นฟีเจอร์ ดังนั้น เราจำเป็นต้องปรับฟังก์ชันการฝึกเล็กน้อยเพื่อจัดการกับสิ่งนี้:


In [7]:
net = EmbedClassifier(vocab_size,32,len(classes)).to(device)

def train_epoch_emb(net,dataloader,lr=0.01,optimizer=None,loss_fn = torch.nn.CrossEntropyLoss(),epoch_size=None, report_freq=200):
    optimizer = optimizer or torch.optim.Adam(net.parameters(),lr=lr)
    loss_fn = loss_fn.to(device)
    net.train()
    total_loss,acc,count,i = 0,0,0,0
    for labels,text,off in dataloader:
        optimizer.zero_grad()
        labels,text,off = labels.to(device), text.to(device), off.to(device)
        out = net(text, off)
        loss = loss_fn(out,labels) #cross_entropy(out,labels)
        loss.backward()
        optimizer.step()
        total_loss+=loss
        _,predicted = torch.max(out,1)
        acc+=(predicted==labels).sum()
        count+=len(labels)
        i+=1
        if i%report_freq==0:
            print(f"{count}: acc={acc.item()/count}")
        if epoch_size and count>epoch_size:
            break
    return total_loss.item()/count, acc.item()/count


train_epoch_emb(net,train_loader, lr=4, epoch_size=25000)

3200: acc=0.6153125
6400: acc=0.6615625
9600: acc=0.6932291666666667
12800: acc=0.715078125
16000: acc=0.7270625
19200: acc=0.7382291666666667
22400: acc=0.7486160714285715


(22.771553103007037, 0.7551983365323096)


## การฝังความหมาย: Word2Vec

ในตัวอย่างก่อนหน้านี้ ชั้น embedding ของโมเดลได้เรียนรู้ที่จะจับคำให้อยู่ในรูปแบบเวกเตอร์ อย่างไรก็ตาม การแทนค่านี้ไม่ได้มีความหมายเชิงความหมายมากนัก จะดีมากหากได้เรียนรู้การแทนค่านั้น ที่คำที่มีความหมายใกล้เคียงกันหรือตรงกันจะถูกแทนด้วยเวกเตอร์ที่อยู่ใกล้กันในแง่ของระยะทางเวกเตอร์บางอย่าง (เช่น ระยะ Euclidean)

เพื่อให้ทำเช่นนั้นได้ เราจำเป็นต้องฝึกโมเดล embedding ของเราล่วงหน้าด้วยชุดข้อความขนาดใหญ่ในแบบเฉพาะ วิธีแรกๆ ในการฝึก embedding เชิงความหมายเรียกว่า [Word2Vec](https://en.wikipedia.org/wiki/Word2vec) มันอิงจากสถาปัตยกรรมหลักสองแบบที่ใช้ผลิตการแทนคำแบบกระจาย:

 - **Continuous bag-of-words** (CBoW) — ในสถาปัตยกรรมนี้ เราฝึกโมเดลเพื่อทำนายคำจากบริบทรอบข้าง สมมติว่า ngram $(W_{-2},W_{-1},W_0,W_1,W_2)$ เป้าหมายของโมเดลคือทำนาย $W_0$ จาก $(W_{-2},W_{-1},W_1,W_2)$
 - **Continuous skip-gram** เป็นตรงข้ามกับ CBoW โมเดลจะใช้บริบทของคำรอบข้างเพื่อทำนายคำปัจจุบัน

CBoW รวดเร็วกว่า ในขณะที่ skip-gram ช้ากว่า แต่ทำงานได้ดีกว่าในการแทนคำที่มีความถี่ต่ำ

![รูปภาพแสดงอัลกอริทึมทั้ง CBoW และ Skip-Gram เพื่อแปลงคำเป็นเวกเตอร์.](../../../../../translated_images/th/example-algorithms-for-converting-words-to-vectors.fbe9207a726922f6.webp)

เพื่อทดลองกับ word2vec ที่ฝึกล่วงหน้าบนชุดข้อมูล Google News เราสามารถใช้ไลบรารี **gensim** ด้านล่างนี้คือการหาคำที่มีความหมายใกล้เคียงกับ 'neural'

> **หมายเหตุ:** เมื่อคุณสร้างเวกเตอร์คำครั้งแรก การดาวน์โหลดอาจใช้เวลาสักครู่!


In [8]:
import gensim.downloader as api
w2v = api.load('word2vec-google-news-300')

In [9]:
for w,p in w2v.most_similar('neural'):
    print(f"{w} -> {p}")

neuronal -> 0.7804799675941467
neurons -> 0.7326500415802002
neural_circuits -> 0.7252851724624634
neuron -> 0.7174385190010071
cortical -> 0.6941086649894714
brain_circuitry -> 0.6923246383666992
synaptic -> 0.6699118614196777
neural_circuitry -> 0.6638563275337219
neurochemical -> 0.6555314064025879
neuronal_activity -> 0.6531826257705688


เรายังสามารถคำนวณเวกเตอร์ฝังจากคำ เพื่อใช้ในการฝึกโมเดลการจำแนกประเภท (เราแสดงเฉพาะ 20 ส่วนประกอบแรกของเวกเตอร์เพื่อความชัดเจน):


In [10]:
w2v.word_vec('play')[:20]

array([ 0.01226807,  0.06225586,  0.10693359,  0.05810547,  0.23828125,
        0.03686523,  0.05151367, -0.20703125,  0.01989746,  0.10058594,
       -0.03759766, -0.1015625 , -0.15820312, -0.08105469, -0.0390625 ,
       -0.05053711,  0.16015625,  0.2578125 ,  0.10058594, -0.25976562],
      dtype=float32)

สิ่งที่ยอดเยี่ยมเกี่ยวกับการฝังเชิงความหมายคือคุณสามารถจัดการการเข้ารหัสเวกเตอร์เพื่อเปลี่ยนความหมายได้ ตัวอย่างเช่น เราสามารถขอให้หาคำที่มีการแทนเวกเตอร์ใกล้เคียงกับคำ *king* และ *woman* ให้มากที่สุด และไกลจากคำ *man*:


In [10]:
w2v.most_similar(positive=['king','woman'],negative=['man'])[0]

('queen', 0.7118192911148071)

ทั้ง CBoW และ Skip-Grams เป็นการฝังแบบ “ทำนาย” ที่จะพิจารณาเฉพาะบริบทท้องถิ่นเท่านั้น Word2Vec จึงไม่ได้ใช้ประโยชน์จากบริบทระดับโลก

**FastText** สร้างต่อจาก Word2Vec โดยการเรียนรู้การแทนเวกเตอร์สำหรับแต่ละคำและ n-แกรมของตัวอักษรที่พบในแต่ละคำ ค่าของการแทนเหล่านี้จะถูกเฉลี่ยเป็นเวกเตอร์เดียวในแต่ละขั้นตอนการฝึก ในขณะที่สิ่งนี้เพิ่มการคำนวณมากขึ้นในการฝึกล่วงหน้า แต่จะช่วยให้การฝังคำนั้นเข้ารหัสข้อมูลย่อยของคำได้

อีกวิธีหนึ่งคือ **GloVe** ที่ใช้แนวคิดของเมทริกซ์การเกิดร่วมกัน ใช้วิธีทางประสาทเทียมในการแยกเมทริกซ์การเกิดร่วมกันเป็นเวกเตอร์คำที่แสดงออกและไม่เชิงเส้นมากขึ้น

คุณสามารถทดลองเล่นตัวอย่างโดยเปลี่ยน embeddings เป็น FastText และ GloVe เนื่องจาก gensim รองรับโมเดลการฝังคำที่แตกต่างกันหลายแบบ


## การใช้การฝังตัวที่ผ่านการฝึกไว้ล่วงหน้าใน PyTorch

เราสามารถปรับตัวอย่างข้างต้นเพื่อเติมเต็มเมทริกซ์ในเลเยอร์ embedding ของเราด้วยการฝังตัวเชิงความหมาย เช่น Word2Vec เราต้องพิจารณาว่าคำศัพท์ของการฝังตัวที่ผ่านการฝึกไว้และชุดข้อมูลข้อความของเราอาจไม่ตรงกัน ดังนั้นเราจะเริ่มต้นน้ำหนักของคำที่ขาดหายไปด้วยค่าที่สุ่มขึ้น:


In [11]:
embed_size = len(w2v.get_vector('hello'))
print(f'Embedding size: {embed_size}')

net = EmbedClassifier(vocab_size,embed_size,len(classes))

print('Populating matrix, this will take some time...',end='')
found, not_found = 0,0
for i,w in enumerate(vocab.get_itos()):
    try:
        net.embedding.weight[i].data = torch.tensor(w2v.get_vector(w))
        found+=1
    except:
        net.embedding.weight[i].data = torch.normal(0.0,1.0,(embed_size,))
        not_found+=1

print(f"Done, found {found} words, {not_found} words missing")
net = net.to(device)

Embedding size: 300
Populating matrix, this will take some time...Done, found 41080 words, 54732 words missing


ตอนนี้มาฝึกโมเดลของเรากันสักหน่อย โปรดทราบว่าระยะเวลาที่ใช้ในการฝึกโมเดลนานกว่าตัวอย่างก่อนหน้านี้อย่างมาก เนื่องจากขนาดชั้นฝังตัวที่ใหญ่กว่า และด้วยจำนวนพารามิเตอร์ที่สูงกว่ามากด้วย นอกจากนี้ จากเหตุผลนี้ เราอาจจำเป็นต้องฝึกโมเดลของเราด้วยตัวอย่างที่มากขึ้นหากเราต้องการหลีกเลี่ยงการฟิตเกินไป


In [12]:
train_epoch_emb(net,train_loader, lr=4, epoch_size=25000)

3200: acc=0.6359375
6400: acc=0.68109375
9600: acc=0.7067708333333333
12800: acc=0.723671875
16000: acc=0.73625
19200: acc=0.7463541666666667
22400: acc=0.7560714285714286


(214.1013875559821, 0.7626759436980166)

ในกรณีของเรา เราไม่เห็นการเพิ่มขึ้นอย่างมากของความแม่นยำ ซึ่งน่าจะเป็นเพราะคำศัพท์ที่แตกต่างกันอย่างมาก
เพื่อแก้ไขปัญหาคำศัพท์ที่แตกต่างกัน เราสามารถใช้หนึ่งในวิธีต่อไปนี้:
* ฝึกโมเดล word2vec ใหม่ด้วยคำศัพท์ของเรา
* โหลดชุดข้อมูลของเราด้วยคำศัพท์จากโมเดล word2vec ที่ผ่านการฝึกมาแล้ว คำศัพท์ที่ใช้ในการโหลดชุดข้อมูลสามารถระบุได้ในระหว่างการโหลด

วิธีหลังดูเหมือนจะง่ายกว่า โดยเฉพาะอย่างยิ่งเพราะเฟรมเวิร์ก PyTorch `torchtext` มีการรองรับ embeddings ในตัว เราสามารถสร้างคำศัพท์ที่อิงจาก GloVe ได้ในลักษณะดังนี้:


In [14]:
vocab = torchtext.vocab.GloVe(name='6B', dim=50)

100%|█████████▉| 399999/400000 [00:15<00:00, 25411.14it/s]


พจนานุกรมที่โหลดมีการดำเนินการพื้นฐานดังนี้:
* `vocab.stoi` ช่วยให้เราสามารถแปลงคำเป็นดัชนีในพจนานุกรม
* `vocab.itos` เป็นการทำงานในทางกลับกัน - แปลงตัวเลขเป็นคำ
* `vocab.vectors` เป็นอาเรย์ของเวกเตอร์ฝังตัว ดังนั้นเพื่อรับเวกเตอร์ฝังตัวของคำ `s` เราต้องใช้ `vocab.vectors[vocab.stoi[s]]`

นี่คือตัวอย่างการจัดการเวกเตอร์ฝังตัวเพื่อสาธิตสมการ **kind-man+woman = queen** (ฉันต้องปรับค่าสัมประสิทธิ์เล็กน้อยเพื่อให้มันทำงานได้):


In [15]:
# get the vector corresponding to kind-man+woman
qvec = vocab.vectors[vocab.stoi['king']]-vocab.vectors[vocab.stoi['man']]+1.3*vocab.vectors[vocab.stoi['woman']]
# find the index of the closest embedding vector 
d = torch.sum((vocab.vectors-qvec)**2,dim=1)
min_idx = torch.argmin(d)
# find the corresponding word
vocab.itos[min_idx]

'queen'

ในการฝึกตัวจำแนกโดยใช้การฝังเหล่านั้น เราต้องเข้ารหัสชุดข้อมูลของเราก่อนโดยใช้คำศัพท์ของ GloVe:


In [16]:
def offsetify(b):
    # first, compute data tensor from all sequences
    x = [torch.tensor(encode(t[1],voc=vocab)) for t in b] # pass the instance of vocab to encode function!
    # now, compute the offsets by accumulating the tensor of sequence lengths
    o = [0] + [len(t) for t in x]
    o = torch.tensor(o[:-1]).cumsum(dim=0)
    return ( 
        torch.LongTensor([t[0]-1 for t in b]), # labels
        torch.cat(x), # text 
        o
    )

อย่างที่เราเห็นข้างต้น เอ็มเบดดิ้งเวกเตอร์ทั้งหมดจะถูกเก็บไว้ในแมทริกซ์ `vocab.vectors` ซึ่งทำให้การโหลดน้ำหนักเหล่านั้นเข้ากับน้ำหนักของเลเยอร์ embedding เป็นเรื่องง่ายมากด้วยการคัดลอกอย่างง่าย:


In [17]:
net = EmbedClassifier(len(vocab),len(vocab.vectors[0]),len(classes))
net.embedding.weight.data = vocab.vectors
net = net.to(device)

ตอนนี้มาฝึกโมเดลของเราและดูว่าเราจะได้ผลลัพธ์ที่ดีขึ้นหรือไม่:


In [18]:
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=16, collate_fn=offsetify, shuffle=True)
train_epoch_emb(net,train_loader, lr=4, epoch_size=25000)

3200: acc=0.6271875
6400: acc=0.68078125
9600: acc=0.7030208333333333
12800: acc=0.71984375
16000: acc=0.7346875
19200: acc=0.7455729166666667
22400: acc=0.7529464285714286


(35.53972978646833, 0.7575175943698017)

หนึ่งในเหตุผลที่เราไม่เห็นการเพิ่มขึ้นอย่างมีนัยสำคัญในความแม่นยำเนื่องมาจากคำบางคำจากชุดข้อมูลของเราหายไปในพจนานุกรม GloVe ที่ผ่านการฝึกอบรมล่วงหน้า และดังนั้นคำเหล่านั้นจึงถูกมองข้ามโดยปริยาย เพื่อเอาชนะข้อเท็จจริงนี้ เราสามารถฝึกการฝังตัวของเราเองบนชุดข้อมูลของเราได้


## การฝังตัวแบบมีบริบท

ข้อจำกัดสำคัญอย่างหนึ่งของการแทนค่าการฝังตัวที่ผ่านการฝึกล่วงหน้ารูปแบบดั้งเดิม เช่น Word2Vec คือปัญหาการจำแนกความหมายของคำ แม้ว่าการฝังตัวที่ผ่านการฝึกล่วงหน้าจะสามารถจับความหมายบางส่วนของคำในบริบทได้ แต่ความหมายทุกความหมายของคำจะถูกเข้ารหัสลงในการฝังตัวเดียวกัน ซึ่งอาจก่อให้เกิดปัญหาในโมเดลภายหลัง เนื่องจากหลายคำ เช่น คำว่า 'play' จะมีความหมายต่างกันขึ้นอยู่กับบริบทที่นำมาใช้

ตัวอย่างเช่น คำว่า 'play' ในสองประโยคนี้มีความหมายแตกต่างกันมาก:
- ฉันไปดู **ละคร** ที่โรงละคร
- จอห์นต้องการ **เล่น** กับเพื่อนของเขา

การฝังตัวที่ผ่านการฝึกล่วงหน้าข้างต้นเป็นตัวแทนของความหมายทั้งสองนี้ของคำว่า 'play' ในการฝังตัวเดียวกัน เพื่อเอาชนะข้อจำกัดนี้ เราจำเป็นต้องสร้างการฝังตัวขึ้นบนพื้นฐานของ **โมเดลภาษา** ซึ่งถูกฝึกบนชุดข้อมูลข้อความขนาดใหญ่ และ *รู้* ว่าคำต่าง ๆ สามารถนำมาประกอบกันในบริบทที่แตกต่างกันได้อย่างไร การพูดคุยเกี่ยวกับการฝังตัวแบบมีบริบทยังไม่อยู่ในขอบเขตของบทแนะนำนี้ แต่เราจะกลับมาพูดถึงเมื่อพูดถึงโมเดลภาษาในหน่วยต่อไป


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ปฏิเสธความรับผิดชอบ**:
เอกสารนี้ได้รับการแปลโดยใช้บริการแปลภาษา AI [Co-op Translator](https://github.com/Azure/co-op-translator) ขณะที่เราพยายามให้ความถูกต้อง โปรดทราบว่าการแปลโดยอัตโนมัติอาจมีข้อผิดพลาดหรือความไม่ถูกต้อง เอกสารต้นฉบับในภาษาต้นทางควรถูกพิจารณาเป็นแหล่งข้อมูลที่เชื่อถือได้ สำหรับข้อมูลที่สำคัญ แนะนำให้ใช้การแปลโดยมนุษย์มืออาชีพ เราไม่รับผิดชอบต่อความเข้าใจผิดหรือการตีความที่ผิดพลาดที่เกิดขึ้นจากการใช้การแปลนี้
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
